# Math I: Probability for security

Almost every model in this course is a probability model, and almost every mistake in security ML is a probability mistake
(ignoring base rates, assuming independence, trusting a point estimate). This notebook is a *hands-on* refresher: every concept
is checked by **simulation** as well as by formula.

Contents: events and simulation, conditional probability and Bayes' theorem (with the *base-rate fallacy*), independence (the
"naive" in Naive Bayes), random variables and distributions, estimation and smoothing, entropy.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import stats

plt.rcParams["figure.dpi"] = 100
rng = np.random.default_rng(42)

## 1. Events, probabilities and simulation

A **sample space** $S$ is the set of outcomes, an **event** $A\subseteq S$ a subset, and a **probability** assigns a number
$0\le P(A)\le 1$ with $P(S)=1$ and $P(A\cup B)=P(A)+P(B)-P(A\cap B)$. The complement satisfies $P(A^c)=1-P(A)$.

**Law of large numbers:** the relative frequency of an event converges to its probability. Simulation is therefore a
*universal calculator*: when the algebra is hard, simulate.

In [ ]:
n = 20_000
dice = rng.integers(1, 7, size=(n, 2))  # two dice
total = dice.sum(axis=1)
event = total >= 10  # A = "sum is at least 10": outcomes (4,6),(5,5),(6,4),(5,6),(6,5),(6,6) -> 6/36
running = np.cumsum(event) / np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(running, lw=1)
ax.axhline(6 / 36, color="k", ls="--", label="exact 6/36")
ax.set(xscale="log", xlabel="number of throws", ylabel="relative frequency of A")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Conditional probability and Bayes' theorem

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)},\qquad
P(A\mid B)=\frac{P(B\mid A)\,P(A)}{P(B)},\qquad
P(B)=\sum_i P(B\mid A_i)P(A_i)
$$

### The base-rate fallacy

An intrusion detector flags 99% of attacks (*true-positive rate*) and raises a false alarm on 1% of benign events. Attacks
are rare: 1 event in 1,000. **If the alarm rings, how likely is it an attack?**

In [ ]:
tpr, fpr, prevalence = 0.99, 0.01, 0.001
p_alarm = tpr * prevalence + fpr * (1 - prevalence)
posterior = tpr * prevalence / p_alarm
print(f"P(attack | alarm) = {posterior:.3f}")

# check by simulation
N = 2_000_000
attack = rng.random(N) < prevalence
alarm = np.where(attack, rng.random(N) < tpr, rng.random(N) < fpr)
print(f"simulated         = {attack[alarm].mean():.3f}   ({alarm.sum()} alarms, {attack[alarm].sum()} real)")

About **9%**: roughly ten alarms per real attack. What matters is the **precision at the real prevalence**, not the detection rate.
The plot shows how the posterior depends on the prevalence and on the false-positive rate.

In [ ]:
prev = np.logspace(-5, -0.5, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for f in (0.1, 0.01, 0.001, 0.0001):
    ax.plot(prev, tpr * prev / (tpr * prev + f * (1 - prev)), label=f"false-alarm rate {f:g}")
ax.set(xscale="log", xlabel="prevalence of attacks", ylabel="P(attack | alarm)  (precision)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Independence: the "naive" in Naive Bayes

$A$ and $B$ are independent if $P(A\cap B)=P(A)P(B)$. Naive Bayes assumes that the words of a message are independent **given the
class**. How wrong is that on real SMS data? For pairs of words we compare $P(a,b\mid\text{spam})$ with $P(a\mid\text{spam})P(b\mid\text{spam})$.

In [ ]:
df = pd.read_csv("../../datasets/spam.csv").dropna().drop_duplicates(subset="SMS")
spam = df[df.Target == "spam"].SMS.str.lower()
print(f"{len(spam)} unique spam messages")


def has(word):
    return spam.str.contains(rf"\b{word}\b", regex=True).to_numpy()


print(f"{'pair':22}{'P(a,b|spam)':>13}{'P(a)P(b)':>11}{'ratio':>8}")
for a, b in [("free", "call"), ("free", "win"), ("claim", "prize"), ("call", "now"), ("to", "you"), ("txt", "stop")]:
    pa, pb, pab = has(a).mean(), has(b).mean(), (has(a) & has(b)).mean()
    print(f"{a + ' & ' + b:22}{pab:13.3f}{pa * pb:11.3f}{pab / (pa * pb):8.1f}")

A ratio of 1 would mean independence; `claim` & `prize` co-occur several times more than chance. Naive Bayes therefore
**counts the same evidence twice** and produces over-confident posteriors, but still ranks messages well.

## 4. Random variables and distributions

A **random variable** maps outcomes to numbers. The ones you will meet most:

| distribution | models | security example |
|:--|:--|:--|
| Bernoulli($p$) | one yes/no event | is this message spam? |
| Binomial($n,p$) | number of successes in $n$ trials | false alarms among $n$ benign events |
| Poisson($\lambda$) | rare events per interval | failed logins per minute |
| Gaussian($\mu,\sigma^2$) | sums of many small effects | packet sizes, sensor noise (anomaly = far from $\mu$) |

In [ ]:
n_events, fpr = 100_000, 0.01
k = np.arange(850, 1150)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar(k, stats.binom.pmf(k, n_events, fpr), width=1.0, label="Binomial(100000, 0.01)")
ax[0].plot(
    k, stats.norm.pdf(k, n_events * fpr, np.sqrt(n_events * fpr * (1 - fpr))), "r", label="Gaussian approximation"
)
ax[0].set(title="false alarms per day", xlabel="alarms")
ax[0].legend()
lam = 3.0
kk = np.arange(0, 12)
ax[1].bar(kk, stats.poisson.pmf(kk, lam))
ax[1].set(title=f"Poisson({lam:g}): failed logins per minute", xlabel="count")
plt.tight_layout()
plt.show()
print(f"expected false alarms/day: {n_events * fpr:.0f} +- {np.sqrt(n_events * fpr * (1 - fpr)):.0f}")

### Expectation, variance and the z-score

$E[X]=\sum_x x\,P(x)$, $\operatorname{Var}(X)=E[(X-E[X])^2]$. A **z-score** $z=(x-\mu)/\sigma$ measures how many standard
deviations a value is from the mean: the simplest anomaly detector flags $|z|>3$, which for a Gaussian happens with probability
$0.27\%$. With millions of events per day, *even that flags thousands*.

In [ ]:
p_out = 2 * stats.norm.sf(3)
print(f"P(|z|>3) = {p_out:.4f}  -> {p_out * 5_000_000:.0f} alarms per 5M benign events")

### Central limit theorem

The mean of many independent samples is approximately Gaussian, whatever the original distribution. That is why Gaussian
assumptions are often reasonable for *aggregated* features (average flow size per host) and wrong for raw heavy-tailed ones.

In [ ]:
skewed = rng.exponential(1.0, size=(20_000, 30))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(skewed[:, 0], bins=60, density=True)
ax[0].set_title("one sample: Exponential(1)")
ax[1].hist(skewed.mean(axis=1), bins=60, density=True)
xs = np.linspace(0.4, 1.8, 200)
ax[1].plot(xs, stats.norm.pdf(xs, 1, 1 / np.sqrt(30)), "r")
ax[1].set_title("mean of 30 samples")
plt.tight_layout()
plt.show()

## 5. Estimation, smoothing and priors

**Maximum likelihood**: choose the parameter that makes the observed data most probable. For Bernoulli data, $\hat p=k/n$. With
$k=0$ spam messages containing a word we would conclude $P(\text{word}\mid\text{spam})=0$: a *single unseen word vetoes the
whole message*. A **Beta($\alpha,\beta$) prior** encodes a belief before seeing data; the posterior mean is
$\dfrac{k+\alpha}{n+\alpha+\beta}$. With $\alpha=\beta=1$ this is **Laplace smoothing**, the $(k+1)/(n+2)$ used by Naive Bayes.

In [ ]:
k_obs, n_obs = 0, 5
x = np.linspace(0, 1, 400)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(x, stats.beta.pdf(x, 1 + k_obs, 1 + n_obs - k_obs), label="posterior, Beta(1,1) prior (Laplace)")
ax.plot(x, stats.beta.pdf(x, 0.1 + k_obs, 0.1 + n_obs - k_obs), label="posterior, Beta(0.1,0.1) prior")
ax.axvline(k_obs / n_obs, color="k", ls=":", label="MLE = 0")
ax.set(xlabel="p = P(word | spam)", ylim=(0, 8))
ax.legend()
plt.tight_layout()
plt.show()
print(f"posterior mean with Laplace smoothing: {(k_obs + 1) / (n_obs + 2):.3f}")

## 6. Entropy and cross-entropy

The **entropy** $H(p)=-\sum_x p(x)\log_2 p(x)$ is the average number of bits needed to encode a symbol from $p$. The
**cross-entropy** $H(p,q)=-\sum_x p(x)\log_2 q(x)$ is the cost of encoding with the *wrong* model $q$; the logistic-regression
loss is exactly this quantity. The excess, $D_{KL}(p\Vert q)=H(p,q)-H(p)\ge0$, measures how wrong $q$ is.

In [ ]:
def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


for name, p in [
    ("fair coin", [0.5, 0.5]),
    ("biased coin 0.9/0.1", [0.9, 0.1]),
    ("99.9% benign", [0.999, 0.001]),
    ("uniform bytes", [1 / 256] * 256),
]:
    print(f"{name:22} H = {entropy(p):.3f} bits")

High entropy = unpredictable. Encrypted or packed malware has near-maximal byte entropy (8 bits), which is why entropy is a
classic static feature (see `notebook_07`).

## Exercises

1. Change the prevalence in section 2 to 1 in 100 and to 1 in 100,000. Explain the two results. What false-alarm rate is needed for 90% precision at 1 in 100,000?
2. Two independent detectors each miss 10% of attacks. What is the probability that *both* miss? What if their misses are perfectly correlated?
3. Simulate the sum of 12 uniform random numbers on $[0,1]$ and check that it is nearly Gaussian ($\mu=6$, $\sigma=1$).
4. For the pair `free`/`call` compute $P(a,b\mid\text{ham})$. Is the independence assumption better or worse in the ham class?